# Error Analysis — Multichannel Sarcasm Detection — Google Colab

> **Jalankan notebook ini SETELAH training selesai** (`colab_train_multichannel.ipynb`).
> Checkpoint `best_model.pt` harus sudah tersedia di `checkpoints/{dataset}/`.

**Cara pakai:**
1. Pastikan `colab_train_multichannel.ipynb` sudah selesai dijalankan
2. Pilih **Runtime → Change runtime type → T4 GPU**
3. Jalankan cell dari atas ke bawah

**Output yang dihasilkan (otomatis tersimpan ke Google Drive):**
```
My Drive/multi_channel_method/error_analysis/
├── reddit_false_positive.csv
├── reddit_false_negative.csv
├── reddit_summary.txt
├── twitter_false_positive.csv
├── twitter_false_negative.csv
└── twitter_summary.txt
```

> **Kenapa otomatis ke Drive?** Karena Cell 2 mengatur working directory ke folder Drive.
> Semua file yang ditulis oleh script langsung masuk ke Drive tanpa perlu copy manual.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Set Working Directory

> **Ubah `PROJECT_PATH`** jika folder-mu tidak ada di `My Drive/multi_channel_method`.

In [ ]:
import os

PROJECT_PATH = '/content/drive/MyDrive/multi_channel_method'
os.chdir(PROJECT_PATH)
print('Working dir:', os.getcwd())
print('Files found:', os.listdir('.'))

## Cell 3 — Install Dependencies & Cek GPU

In [ ]:
!pip install -q --upgrade transformers sentencepiece
!nvidia-smi

## Cell 4 — Verifikasi Checkpoint dan Data

In [ ]:
import os

all_ok = True

# Cek checkpoint
for dataset in ['reddit', 'twitter']:
    ckpt = f'checkpoints/{dataset}/best_model.pt'
    status = 'OK' if os.path.exists(ckpt) else 'MISSING!'
    if status == 'MISSING!':
        all_ok = False
    print(f'Checkpoint [{dataset}]: {status}')

print()

# Cek data test dan train (dibutuhkan untuk rebuild vocab)
for dataset in ['reddit', 'twitter']:
    for split in ['train', 'test']:
        for ext in ['.csv', '.csv.graph.new', '.csv.sentic']:
            path = f'real_data/{dataset}/{split}{ext}'
            status = 'OK' if os.path.exists(path) else 'MISSING!'
            if status == 'MISSING!':
                all_ok = False
            print(f'[{dataset}] {split}{ext}: {status}')

print()
if all_ok:
    print('Semua file tersedia. Siap error analysis!')
else:
    print('Ada file yang hilang! Pastikan training sudah selesai dan data tersedia.')

## Cell 5 — Buat Direktori Output

In [ ]:
import os

os.makedirs('error_analysis', exist_ok=True)
os.makedirs('logs', exist_ok=True)
print('Direktori output siap.')

---
## DATASET: REDDIT

## Cell 6 — Jalankan Error Analysis: Reddit

In [ ]:
!python error_analysis.py \
    --test_data        real_data/reddit/test.csv \
    --test_split_name  test.csv \
    --graph_dir        real_data/reddit/ \
    --checkpoint_dir   checkpoints/reddit \
    --dataset_name     reddit \
    --train_data       real_data/reddit/train.csv \
    --train_split_name train.csv \
    --train_graph_dir  real_data/reddit/ \
    --batch_size       32 \
    --output_dir       error_analysis/ \
    2>&1 | tee logs/error_analysis_reddit.log

## Cell 7 — Lihat Ringkasan Statistik: Reddit

In [ ]:
with open('error_analysis/reddit_summary.txt', 'r', encoding='utf-8') as f:
    print(f.read())

## Cell 8 — Baca & Tampilkan FP dan FN Reddit

In [ ]:
import pandas as pd

pd.set_option('display.max_colwidth', 80)

fp_reddit = pd.read_csv('error_analysis/reddit_false_positive.csv')
fn_reddit = pd.read_csv('error_analysis/reddit_false_negative.csv')

print(f'False Positive (Reddit): {len(fp_reddit)} baris')
print(f'False Negative (Reddit): {len(fn_reddit)} baris')
print(f'Kolom: {list(fp_reddit.columns)}')
print()
print('--- 5 FP teratas (confidence tinggi = model sangat yakin tapi salah) ---')
display(fp_reddit.sort_values('confidence_score', ascending=False).head())
print()
print('--- 5 FN teratas (confidence rendah = borderline / hard case) ---')
display(fn_reddit.sort_values('confidence_score', ascending=True).head())

## Cell 9 — Lokasi CSV Reddit di Drive

File sudah otomatis tersimpan ke Drive karena working directory di-set ke folder Drive di Cell 2.
Tidak perlu copy manual.

In [ ]:
import os

files_reddit = [
    'error_analysis/reddit_false_positive.csv',
    'error_analysis/reddit_false_negative.csv',
    'error_analysis/reddit_summary.txt',
]

print('File Reddit tersimpan di Google Drive:')
for f in files_reddit:
    drive_path = f'My Drive/multi_channel_method/{f}'
    size_kb    = os.path.getsize(f) / 1024
    print(f'  {drive_path}  ({size_kb:.1f} KB)')

## Cell 10 — (Opsional) Download CSV Reddit ke Komputer Lokal

Jalankan cell ini jika ingin mengunduh file ke komputer, bukan hanya menyimpan di Drive.

In [ ]:
from google.colab import files

files.download('error_analysis/reddit_false_positive.csv')
files.download('error_analysis/reddit_false_negative.csv')

---
## DATASET: TWITTER

## Cell 11 — Jalankan Error Analysis: Twitter

In [ ]:
!python error_analysis.py \
    --test_data        real_data/twitter/test.csv \
    --test_split_name  test.csv \
    --graph_dir        real_data/twitter/ \
    --checkpoint_dir   checkpoints/twitter \
    --dataset_name     twitter \
    --train_data       real_data/twitter/train.csv \
    --train_split_name train.csv \
    --train_graph_dir  real_data/twitter/ \
    --batch_size       32 \
    --output_dir       error_analysis/ \
    2>&1 | tee logs/error_analysis_twitter.log

## Cell 12 — Lihat Ringkasan Statistik: Twitter

In [ ]:
with open('error_analysis/twitter_summary.txt', 'r', encoding='utf-8') as f:
    print(f.read())

## Cell 13 — Baca & Tampilkan FP dan FN Twitter

In [ ]:
import pandas as pd

pd.set_option('display.max_colwidth', 80)

fp_twitter = pd.read_csv('error_analysis/twitter_false_positive.csv')
fn_twitter = pd.read_csv('error_analysis/twitter_false_negative.csv')

print(f'False Positive (Twitter): {len(fp_twitter)} baris')
print(f'False Negative (Twitter): {len(fn_twitter)} baris')
print(f'Kolom: {list(fp_twitter.columns)}')
print()
print('--- 5 FP teratas ---')
display(fp_twitter.sort_values('confidence_score', ascending=False).head())
print()
print('--- 5 FN teratas ---')
display(fn_twitter.sort_values('confidence_score', ascending=True).head())

## Cell 14 — Lokasi CSV Twitter di Drive

In [ ]:
import os

files_twitter = [
    'error_analysis/twitter_false_positive.csv',
    'error_analysis/twitter_false_negative.csv',
    'error_analysis/twitter_summary.txt',
]

print('File Twitter tersimpan di Google Drive:')
for f in files_twitter:
    drive_path = f'My Drive/multi_channel_method/{f}'
    size_kb    = os.path.getsize(f) / 1024
    print(f'  {drive_path}  ({size_kb:.1f} KB)')

## Cell 15 — (Opsional) Download CSV Twitter ke Komputer Lokal

In [ ]:
from google.colab import files

files.download('error_analysis/twitter_false_positive.csv')
files.download('error_analysis/twitter_false_negative.csv')

---
## Cell 16 — (Opsional) Lihat Log Jika Ada Error

In [ ]:
# Ganti 'reddit' dengan 'twitter' sesuai kebutuhan
!tail -50 logs/error_analysis_reddit.log